In [3]:
#import the shakespeare dataset
#pip is python package installer
%pip install wget
#wget is package that downalods files for us
import wget
wget.download('https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt')

Note: you may need to restart the kernel to use updated packages.
100% [......................................................] 1115394 / 1115394

'input (2).txt'

In [5]:
with open('input.txt','r',encoding = 'utf-8') as file:
    text = file.read()

In [7]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [9]:
print('The length of the characters in the text: ', len(text))

The length of the characters in the text:  1115394


In [11]:
#Create the tokenizention so the gpt can understand prompts and has a libray to write on
#join takes the items in an iterable and joins them by a speerator the one we choose was no space 
vocab = sorted(list(set(text)))
vocab_size  = len(vocab)
print(vocab)
print('The length of the vocabucalry is : ',vocab_size )

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
The length of the vocabucalry is :  65


In [13]:
#since our transformers cant understand strings we need to be able to assign an integer to each token
#allowing us to convert string inputs into integers to which the transformer can read
#stoi - String to integer
#itos - integer to String
stoi = {}
for i in range(0,vocab_size):
    stoi[vocab[i]] = i
itos = {}
for i in range(0,vocab_size):
    itos[i] = vocab[i]
def encode(input):
    output = []
    for letter in input:
        output.append(stoi[letter])
    return output
def decode(input):
    output = ''
    for num in input:
        output += itos[num]
    return output


In [15]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [17]:
import torch
#encode the entire text and store it into a pytorch tensor
#pytorch tensors are just like numpy arrays but they have built in functions that allow us to compute gradients
#and other machine learning things easier and efficently allowing us to use the GPU too
data = torch.tensor(encode(text),dtype = torch.long)
print(data.shape,data.dtype)
print(data[0:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [19]:
#spilt up into training set and valdiation set
#first 90% is going to be training and the rest is validation 
n = int(.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [21]:
torch.manual_seed(1337)
batch_size = 4 # how many blocks will we train in parallel
block_size = 8 # context length for each block

def get_batch(spilt):
    #which dataset we want to use
    if spilt == 'train':
        data = train_data
    else:
        data = val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y
xb,yb = get_batch(train_data)
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)
    

torch.Size([4, 8])
tensor([[ 6,  1, 52, 53, 58,  1, 58, 47],
        [ 6,  1, 54, 50, 39, 52, 58, 43],
        [ 1, 58, 46, 47, 57,  1, 50, 47],
        [ 0, 32, 46, 43, 56, 43,  1, 42]])
targets
torch.Size([4, 8])
tensor([[ 1, 52, 53, 58,  1, 58, 47, 50],
        [ 1, 54, 50, 39, 52, 58, 43, 58],
        [58, 46, 47, 57,  1, 50, 47, 60],
        [32, 46, 43, 56, 43,  1, 42, 53]])


In [161]:
# now we build the MLP that will work on predicting stuff 
n_embed = 32
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLangaugeModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__() # call the parents intializer
        #each token reads from the embedding table
        self.token_embedding_table = nn.Embedding(vocab_size,n_embed)
        #adding a positional embedding 
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        #adding a linear layer
        self.blocks = nn.Sequential(
            Block(n_embed , n_head = 4),
            Block(n_embed , n_head = 4),
            Block(n_embed , n_head = 4)
        )
        self.lm_head = nn.Linear(n_embed, vocab_size)
        
        
        
      


    def forward(self,idx,targets = None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx) #(B,T,C) Batch by time by channel
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        logits = self.lm_head(x)
        if targets == None:
            loss = None
        else :
            B, T, C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss  = F.cross_entropy(logits,targets) # measures the quality of the logits correctly predicting targets
        return logits,loss

        
    def generate(self,idx,max_new_tokens):
        #idx is (B,T) array of indices in the current context 
        for _ in range(max_new_tokens): # how many tokens we want to generate

            #crop idx to the block_size
            idx_cond = idx[:-block_size]
            logits,loss = self(idx_cond) # get the randomized probababilities
            #focus on only on the last token in that example 
            logits = logits[:,-1,:] # becomes (B,C)
            #apply softmax to get probabilites
            probs = F.softmax(logits,dim = -1)
            idx_next = torch.multinomial(probs,num_samples = -1) # from the batch we predict the next token)
            #append sampled index to current sequence
            idx  = torch.cat((idx,idx_next), dim = 1) #(B,T+1)
        return idx
            
            

m = BigramLangaugeModel(vocab_size)
logit,loss = m(xb,yb)


In [163]:
# craeting a pytorch optimizer that can optimize our network 
optimizer  = torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [165]:
#now going to create the training loop where we will optimize the netwrok
batch_size = 32 # increasing batch_size so we can train the model faster 

for steps in range(10000):
    #sample a batch of data 
    xb,yb = get_batch('train')

    #evalutate the loss 
    logits,loss = m(xb,yb)
    optimizer.zero_grad(set_to_none = True) # clears the gradients so bascially the resets the amount we changed the parameters
    # to 0 so the change doesnt pile up
    loss.backward() 
    optimizer.step()

print(loss.item())

1.8454877138137817


In [167]:
# lets create the evalute loss function that will help us better estimate loss than just random batches
eval_iters = 400
@torch.no_grad # tells pytorch we wont be doing any gradient descent calculations helps save memory
def estimate_loss():
    out = {}
    m.eval() # this can turn on or off some layers and switch the modes of the model 
    for spilt in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(spilt)
            logits, loss = m(X,Y)
            losses[k] = loss.item()
        out[spilt] = losses.mean()
        m.train()
    return out

In [169]:
losses = estimate_loss()
print(losses)

{'train': tensor(1.9087), 'val': tensor(2.0373)}


In [78]:
# self- attention block 

class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_embed,head_size,bias = False)
        self.query = nn.Linear(n_embed,head_size, bias = False)
        self.value = nn.Linear(n_embed,head_size, bias = False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        #compute attention scores("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei,dim = -1)
        #perform the wegihted aggregation 
        v = self.value(x)
        out = wei @ v
        return out 
        
    

In [159]:
class MultiHeadAttention(nn.Module):
    def __init__(self,head_size,num_heads):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.proj(out)
        return out
        
        

In [152]:
class FeedForward(nn.Module):
    def __init__(self,n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed,4*n_embed),
            nn.ReLU(),
            nn.Linear(4*n_embed,n_embed)
        )
    def forward(self,x):
        return self.net(x)
        

In [171]:
class Block(nn.Module):
    #Transformer block 
    def __init__(self,n_embed,n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head,head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)


    def forward(self,x):
        #this additiion assignment is the root of residual connections
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
    
        
        